In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ─── Google Drive paths ───────────────────────────────────────────────────────
BASE    = "/content/drive/MyDrive/voice_ai_project"
RUNTIME = "/content/runtime_models"

# ─── LLM model sources (Google Drive) ────────────────────────────────────────
LLM1_SRC = f"{BASE}/models/llm/qwen2.5-1.5b-q4_k_m/qwen2.5-1.5b-instruct-q4_k_m.gguf"
LLM2_SRC = f"{BASE}/models/llm/llama-3.2-3b-q4_k_m/Llama-3.2-3B-Instruct-Q4_K_M.gguf"
LLM3_SRC = f"{BASE}/models/llm/qwen2.5-3b-q4_k_m/qwen2.5-3b-instruct-q4_k_m.gguf"

# ─── Benchmark dataset paths (Google Drive → local SSD) ──────────────────────
MMLU_DRIVE_DIR = f"{BASE}/data/benchmarks/llm"
LOCAL_BENCH    = "/content/runtime_bench"

print("Paths configured.")
print(f"  LLM1 (Qwen1.5B):  {LLM1_SRC}")
print(f"  LLM2 (Llama3B):   {LLM2_SRC}")
print(f"  LLM3 (Qwen3B):    {LLM3_SRC}")


Paths configured.
  LLM1 (Qwen1.5B):  /content/drive/MyDrive/voice_ai_project/models/llm/qwen2.5-1.5b-q4_k_m/qwen2.5-1.5b-instruct-q4_k_m.gguf
  LLM2 (Llama3B):   /content/drive/MyDrive/voice_ai_project/models/llm/llama-3.2-3b-q4_k_m/Llama-3.2-3B-Instruct-Q4_K_M.gguf
  LLM3 (Qwen3B):    /content/drive/MyDrive/voice_ai_project/models/llm/qwen2.5-3b-q4_k_m/qwen2.5-3b-instruct-q4_k_m.gguf


In [ ]:
import shutil
import time

def copy_to_runtime(src, dst):
    start = time.time()
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source not found: {src}")

    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

    elapsed = time.time() - start
    print(f"Copied:\n  {src}\n-> {dst}\nTime: {elapsed:.2f}s")

In [ ]:
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
LLM1_DST = f"{RUNTIME}/llm/qwen2.5-1.5b-instruct-q4_k_m.gguf"
copy_to_runtime(LLM1_SRC, LLM1_DST)

Copied:
  /content/drive/MyDrive/voice_ai_project/models/llm/qwen2.5-1.5b-q4_k_m/qwen2.5-1.5b-instruct-q4_k_m.gguf
-> /content/runtime_models/llm/qwen2.5-1.5b-instruct-q4_k_m.gguf
Time: 29.73s


In [ ]:
LLM3_DST = f"{RUNTIME}/llm/qwen2.5-3b-instruct-q4_k_m.gguf"
copy_to_runtime(LLM3_SRC, LLM3_DST)


Copied:
  /content/drive/MyDrive/voice_ai_project/models/llm/qwen2.5-3b-q4_k_m/qwen2.5-3b-instruct-q4_k_m.gguf
-> /content/runtime_models/llm/qwen2.5-3b-instruct-q4_k_m.gguf
Time: 30.73s


In [ ]:
LLM2_DST = f"{RUNTIME}/llm/Llama-3.2-3B-Instruct-Q4_K_M.gguf"
copy_to_runtime(LLM2_SRC, LLM2_DST)

Copied:
  /content/drive/MyDrive/voice_ai_project/models/llm/llama-3.2-3b-q4_k_m/Llama-3.2-3B-Instruct-Q4_K_M.gguf
-> /content/runtime_models/llm/Llama-3.2-3B-Instruct-Q4_K_M.gguf
Time: 41.17s


## 1.5 Download Qwen2.5-3B Model to Google Drive
Run this cell **once** (CPU runtime is fine, no GPU needed).  
Downloads `qwen2.5-3b-instruct-q4_k_m.gguf` (~2 GB) from Hugging Face to your Drive.


In [ ]:
# ===== Download Qwen2.5-3B-Instruct Q4_K_M to Drive =====
# Run once on any runtime (CPU is fine). Skips if already downloaded.
import os
from huggingface_hub import hf_hub_download

QWEN3B_DIR  = f"{BASE}/models/llm/qwen2.5-3b-q4_k_m"
QWEN3B_FILE = "qwen2.5-3b-instruct-q4_k_m.gguf"
QWEN3B_PATH = f"{QWEN3B_DIR}/{QWEN3B_FILE}"

if os.path.exists(QWEN3B_PATH):
    print(f"Already exists: {QWEN3B_PATH}")
else:
    os.makedirs(QWEN3B_DIR, exist_ok=True)
    print(f"Downloading {QWEN3B_FILE} from Hugging Face...")
    hf_hub_download(
        repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
        filename=QWEN3B_FILE,
        local_dir=QWEN3B_DIR,
    )
    print(f"Saved to {QWEN3B_PATH}")


## 2.0 Download Benchmark Datasets

In [ ]:
# ===== Download Benchmarks to Drive =====
import os
from datasets import load_dataset

BENCH_BASE = f"{BASE}/data/benchmarks"

# ---- ASR: LibriSpeech test-clean ----
asr_bench_path = f"{BENCH_BASE}/asr/librispeech_clean_test"
if not os.path.exists(asr_bench_path):
    print("Downloading LibriSpeech test-clean...")
    ls_clean = load_dataset("librispeech_asr", "clean", split="test")
    ls_clean.save_to_disk(asr_bench_path)
    print(f"Saved to {asr_bench_path}")
else:
    print(f"LibriSpeech already exists at {asr_bench_path}")

# ---- LLM: MMLU 10 subjects (high school + college) ----
MMLU_SUBJECTS = [
    "high_school_mathematics",
    "high_school_physics",
    "high_school_biology",
    "high_school_world_history",
    "high_school_us_history",
    "college_mathematics",
    "college_physics",
    "college_biology",
    "philosophy",
    "international_law",
]

for subj in MMLU_SUBJECTS:
    p = f"{BENCH_BASE}/llm/mmlu_{subj}"
    if not os.path.exists(p):
        print(f"Downloading MMLU/{subj}...")
        ds = load_dataset("cais/mmlu", subj, split="test")
        ds.save_to_disk(p)
    else:
        print(f"MMLU/{subj} already exists")

# ---- TTS: EmergentTTS-Eval ----
tts_bench_path = f"{BENCH_BASE}/tts/emergenttts_eval"
if not os.path.exists(tts_bench_path):
    print("Downloading EmergentTTS-Eval...")
    try:
        tts_ds = load_dataset("bosonai/EmergentTTS-Eval")
        tts_ds.save_to_disk(tts_bench_path)
    except Exception as e:
        print(f"EmergentTTS-Eval download failed: {e}")
else:
    print(f"EmergentTTS-Eval already exists at {tts_bench_path}")

print("\n=== All benchmarks ready ===")


In [ ]:
# ===== Copy benchmarks to local SSD for fast access =====
import shutil, os

LOCAL_BENCH = "/content/benchmarks"
DRIVE_BENCH = f"{BASE}/data/benchmarks"

if not os.path.exists(LOCAL_BENCH):
    print("Copying benchmarks to local SSD...")
    shutil.copytree(DRIVE_BENCH, LOCAL_BENCH)
    print("Done.")
else:
    print("Benchmarks already on local SSD.")

Copying benchmarks to local SSD...
Done.


## 2.2 LLM Benchmark — MMLU (10-subject subset)
Metric: **Accuracy** (multiple-choice A/B/C/D)

**Models:** Qwen2.5-3B-Instruct (Q4_K_M), Llama-3.2-3B-Instruct (Q4_K_M)

**Strategies:** Zero-shot, Role prompt, Few-shot (3-shot), Few-shot CoT

**Subjects (10):** high_school_{mathematics,physics,biology,world_history,us_history}, college_{mathematics,physics,biology}, philosophy, international_law

In [ ]:
# ===== LLM Benchmark: Common Setup =====
# Usage:
#   Run this cell first before any benchmark cells.
from datasets import load_from_disk
import os, time, re, gc, json as _json
import pandas as pd

MMLU_SUBJECTS = [
    "high_school_mathematics", "high_school_physics", "high_school_biology",
    "high_school_world_history", "high_school_us_history",
    "college_mathematics", "college_physics", "college_biology",
    "philosophy", "international_law",
]

mmlu_data = {}
total_q = 0
for subj in MMLU_SUBJECTS:
    p = f"/content/benchmarks/llm/mmlu_{subj}"
    ds = load_from_disk(p)
    mmlu_data[subj] = ds
    total_q += len(ds)
    print(f"  {subj}: {len(ds)} questions")
print(f"\nTotal MMLU questions: {total_q}")

C = ["A", "B", "C", "D"]
FEW_SHOT_N = 3    # demos kept aside; eval skips these rows
COT_MAX    = 50   # max questions per subject for CoT (speed limit)
N_CTX      = 2048

# ── Prompt builders ───────────────────────────────────────────────────────────

def _mc_block(question, choices, answer=None):
    block = f"Question: {question}\n"
    for i, ch in enumerate(choices):
        block += f"{C[i]}. {ch}\n"
    if answer:
        block += f"Answer: {answer}\n"
    return block

def format_mmlu_prompt(question, choices):
    """Zero-shot: plain question, ask for single letter."""
    return _mc_block(question, choices) + "Answer with just the letter (A, B, C, or D): "

def build_few_shot(question, choices, demos):
    """3-shot: first FEW_SHOT_N samples used as examples.
    demos = [(q, choices_list, answer_idx), ...]
    """
    prompt = "Here are some example questions with answers:\n\n"
    for q, ch, ans_idx in demos:
        prompt += _mc_block(q, ch, C[ans_idx]) + "\n"
    prompt += "Now answer the following question.\n\n"
    prompt += _mc_block(question, choices)
    prompt += "Answer with just the letter (A, B, C, or D): "
    return prompt

# ── Few-shot CoT demos (2 hand-crafted reasoning chains per subject) ──────────
COT_DEMOS = {
    "high_school_mathematics": [
        ("If f(x) = 2x^2 - 3x + 1, what is f(2)?",
         ["1", "3", "5", "7"],
         "Substitute x=2: 2(4)-3(2)+1 = 8-6+1 = 3.", "B"),
        ("A bag has 3 red and 2 blue marbles. P(red)?",
         ["2/5", "3/5", "1/2", "2/3"],
         "Total=5, red=3, P=3/5.", "B"),
    ],
    "high_school_physics": [
        ("Object dropped from rest. Speed after 2s? (g=10 m/s^2)",
         ["5 m/s", "10 m/s", "20 m/s", "40 m/s"],
         "v = u+at = 0+10*2 = 20 m/s.", "C"),
        ("10 kg object, net force 30 N. Acceleration?",
         ["0.3 m/s^2", "3 m/s^2", "30 m/s^2", "300 m/s^2"],
         "a = F/m = 30/10 = 3 m/s^2.", "B"),
    ],
    "high_school_biology": [
        ("Primary site of ATP production in eukaryotes?",
         ["Nucleus", "Ribosome", "Mitochondrion", "Golgi apparatus"],
         "Mitochondria perform oxidative phosphorylation to produce most ATP.", "C"),
        ("Aa x Aa cross: fraction homozygous recessive?",
         ["1/4", "1/2", "3/4", "0"],
         "Aa x Aa gives 1AA:2Aa:1aa. Homozygous recessive aa = 1/4.", "A"),
    ],
    "high_school_world_history": [
        ("Which dynasty established the Silk Road ~130 BCE?",
         ["Roman Empire", "Han Dynasty", "Ottoman Empire", "Mongol Empire"],
         "Han Dynasty of China opened overland trade routes westward ~130 BCE.", "B"),
        ("Industrial Revolution originated in?",
         ["France", "Germany", "Great Britain", "United States"],
         "Began in Great Britain mid-18th century with steam power innovations.", "C"),
    ],
    "high_school_us_history": [
        ("Declaration of Independence primarily drafted by?",
         ["George Washington", "Benjamin Franklin", "John Adams", "Thomas Jefferson"],
         "The Continental Congress chose Thomas Jefferson as principal author in 1776.", "D"),
        ("Which amendment abolished slavery in the US?",
         ["13th", "14th", "15th", "16th"],
         "13th Amendment ratified December 1865 abolished slavery.", "A"),
    ],
    "college_mathematics": [
        ("Derivative of f(x) = x^3 - 4x^2 + 2x - 1?",
         ["3x^2-4x+2", "3x^2-8x+2", "x^2-8x+2", "3x^2-8x+1"],
         "Power rule: d/dx gives 3x^2 - 8x + 2.", "B"),
        ("det([[2,1],[3,4]])?",
         ["5", "8", "11", "14"],
         "det = 2*4 - 1*3 = 5.", "A"),
    ],
    "college_physics": [
        ("Parallel-plate capacitor: d doubled, A unchanged. Capacitance?",
         ["Doubles", "Halves", "Quadruples", "Unchanged"],
         "C = e0*A/d. Doubling d halves C.", "B"),
        ("Bending of light around obstacles?",
         ["Reflection", "Refraction", "Diffraction", "Interference"],
         "Diffraction is the spreading of waves around edges.", "C"),
    ],
    "college_biology": [
        ("Enzyme that unwinds DNA at replication fork?",
         ["DNA polymerase", "Helicase", "Ligase", "Primase"],
         "Helicase breaks hydrogen bonds to unwind DNA strands.", "B"),
        ("Organisms converting solar to chemical energy?",
         ["Consumers", "Decomposers", "Producers", "Carnivores"],
         "Producers fix solar energy via photosynthesis.", "C"),
    ],
    "philosophy": [
        ("Kant categorical imperative: act on maxims that:",
         ["Maximize happiness", "Could be universalized", "Follow natural law", "Benefit oneself"],
         "Act only on principles you could will to be a universal law.", "B"),
        ("Descartes cogito: what cannot be doubted?",
         ["Existence of God", "External world", "Existence of thinking self", "Mathematics"],
         "Doubting itself proves a thinking subject exists: cogito ergo sum.", "C"),
    ],
    "international_law": [
        ("'Pacta sunt servanda' means:",
         ["States may withdraw at will", "Treaties must be performed in good faith",
          "Only powerful states follow", "Treaties expire after 10 years"],
         "Latin: agreements must be kept. Vienna Convention on Law of Treaties.", "B"),
        ("ICJ is principal judicial organ of?",
         ["NATO", "WTO", "United Nations", "EU"],
         "ICJ established by UN Charter 1945 as principal judicial organ of the UN.", "C"),
    ],
}

def build_cot_prompt(question, choices, subject):
    """Few-shot CoT: 2 hand-crafted reasoning examples + target question."""
    demos = COT_DEMOS.get(subject, [])
    prompt = "Here are some example questions solved step by step:\n\n"
    for q, ch, reasoning, ans in demos:
        prompt += _mc_block(q, ch)
        prompt += f"Let me think step by step.\n{reasoning}\nAnswer: {ans}\n\n"
    prompt += "Now answer the following question using step-by-step reasoning.\n\n"
    prompt += _mc_block(question, choices)
    prompt += "Let me think step by step.\n"
    return prompt

# ── Answer extraction ─────────────────────────────────────────────────────────

def extract_answer(text):
    """Zero-shot / few-shot (max_tokens=8).
    Word-boundary regex avoids 'A' in 'ANSWER', 'B' in 'BECAUSE', etc.
    Falls back to first char only if it is a valid A/B/C/D.
    """
    text = text.strip().upper()
    m = re.search(r"\b([ABCD])\b", text)
    if m:
        return m.group(1)
    # Only trust first-char if response is genuinely 1 character
    return text[0] if len(text) == 1 and text[0] in "ABCD" else "X"

def extract_answer_cot(text):
    """CoT output parser.
    Priority: explicit 'Answer: X' -> last 'answer' word + nearby letter -> last letter.
    """
    upper = text.upper()
    m = re.search(r"ANSWER\s*:\s*([ABCD])", upper)
    if m:
        return m.group(1)
    last_ans = upper.rfind("ANSWER")
    if last_ans != -1:
        m2 = re.search(r"\b([ABCD])\b", upper[last_ans:])
        if m2:
            return m2.group(1)
    letters = re.findall(r"\b([ABCD])\b", upper)
    return letters[-1] if letters else "X"

# ── Shared CSV helper ─────────────────────────────────────────────────────────
CSV_PATH = f"{BASE}/outputs/tables/llm_prompt_strategies.csv"
os.makedirs(f"{BASE}/outputs/tables", exist_ok=True)

def save_to_csv(model, strategy, scores, elapsed, truncated=0, total_q_run=0):
    """Append / overwrite one (model, strategy) row in the shared CSV."""
    row = {"model": model, "strategy": strategy, "n_ctx": N_CTX,
           "Overall Acc": round(scores.get("Overall Acc", 0), 3)}
    for subj in MMLU_SUBJECTS:
        row[subj] = round(scores.get(subj, 0), 3)
    row["Time(s)"]       = round(elapsed, 1)
    row["Truncated"]     = truncated
    row["Truncate Rate"] = round(truncated / total_q_run, 3) if total_q_run > 0 else 0.0
    new_df = pd.DataFrame([row]).set_index(["model", "strategy"])
    if os.path.exists(CSV_PATH):
        existing = pd.read_csv(CSV_PATH, index_col=["model", "strategy"])
        key = (model, strategy)
        if key in existing.index:
            existing = existing.drop(index=[key])
        combined = pd.concat([existing, new_df])
    else:
        combined = new_df
    combined.to_csv(CSV_PATH)
    print(f"  [CSV] Saved ({model}, {strategy}) -> {CSV_PATH}")

print("Common setup done.")


  high_school_mathematics: 270 questions
  high_school_physics: 151 questions
  high_school_biology: 310 questions
  high_school_world_history: 237 questions
  high_school_us_history: 204 questions
  college_mathematics: 100 questions
  college_physics: 102 questions
  college_biology: 144 questions
  philosophy: 311 questions
  international_law: 121 questions

Total MMLU questions: 1950
Common setup done.


### LLM Model 1: Qwen2.5-1.5B-Instruct (Q4_K_M)

In [ ]:
# ===== LLM Benchmark: Qwen2.5-3B — Zero-Shot =====
# Run: executes zero-shot across all 10 subjects, saves to CSV.
from llama_cpp import Llama

print("Loading Qwen2.5-1.5B...")
llm3 = Llama(model_path=LLM1_DST, n_ctx=N_CTX, n_threads=4, n_gpu_layers=-1, verbose=False)

scores_zs = {}
correct_total = 0
total = 0
start = time.time()

for subj, ds in mmlu_data.items():
    correct = 0
    n = len(ds)
    print(f"  {subj} (0/{n})", end="", flush=True)
    for qi, sample in enumerate(ds):
        prompt = format_mmlu_prompt(sample["question"], sample["choices"])
        out    = llm3(prompt, max_tokens=8, temperature=0.0)
        pred   = extract_answer(out["choices"][0]["text"])
        gold   = C[sample["answer"]]
        if pred == gold:
            correct += 1
        print(f"\r  {subj}  {qi+1}/{n}  acc={correct/(qi+1):.3f}", end="", flush=True)
    acc = correct / n
    scores_zs[subj] = acc
    correct_total += correct
    total += n
    print(f"\r  {subj}: {acc:.3f} ({correct}/{n})          ")

elapsed = time.time() - start
scores_zs["Overall Acc"] = correct_total / total

del llm3; gc.collect()

save_to_csv("Qwen2.5-1.5B", "zero_shot", scores_zs, elapsed)
print(f"\n=== Qwen2.5-1.5B Zero-Shot: {scores_zs['Overall Acc']:.3f} | {elapsed:.1f}s ===")


Loading Qwen2.5-1.5B...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.289 (78/270)          
  high_school_physics: 0.318 (48/151)          
  high_school_biology: 0.645 (200/310)          
  high_school_world_history: 0.755 (179/237)          
  high_school_us_history: 0.711 (145/204)          
  college_mathematics: 0.330 (33/100)          
  college_physics: 0.304 (31/102)          
  college_biology: 0.625 (90/144)          
  philosophy: 0.592 (184/311)          
  international_law: 0.686 (83/121)          
  [CSV] Saved (Qwen2.5-1.5B, zero_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv

=== Qwen2.5-1.5B Zero-Shot: 0.549 | 99.8s ===


LLM Model 1: Qwen2.5-3B-Instruct (Q4_K_M)

In [ ]:
# ===== LLM Benchmark: Qwen2.5-3B — Zero-Shot =====
# Run: executes zero-shot across all 10 subjects, saves to CSV.
from llama_cpp import Llama

print("Loading Qwen2.5-3B...")
llm3 = Llama(model_path=LLM3_DST, n_ctx=N_CTX, n_threads=4, n_gpu_layers=-1, verbose=False)

scores_zs = {}
correct_total = 0
total = 0
start = time.time()

for subj, ds in mmlu_data.items():
    correct = 0
    n = len(ds)
    print(f"  {subj} (0/{n})", end="", flush=True)
    for qi, sample in enumerate(ds):
        prompt = format_mmlu_prompt(sample["question"], sample["choices"])
        out    = llm3(prompt, max_tokens=8, temperature=0.0)
        pred   = extract_answer(out["choices"][0]["text"])
        gold   = C[sample["answer"]]
        if pred == gold:
            correct += 1
        print(f"\r  {subj}  {qi+1}/{n}  acc={correct/(qi+1):.3f}", end="", flush=True)
    acc = correct / n
    scores_zs[subj] = acc
    correct_total += correct
    total += n
    print(f"\r  {subj}: {acc:.3f} ({correct}/{n})          ")

elapsed = time.time() - start
scores_zs["Overall Acc"] = correct_total / total

del llm3; gc.collect()

save_to_csv("Qwen2.5-3B", "zero_shot", scores_zs, elapsed)
print(f"\n=== Qwen2.5-3B Zero-Shot: {scores_zs['Overall Acc']:.3f} | {elapsed:.1f}s ===")


Loading Qwen2.5-3B...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.396 (107/270)          
  high_school_physics: 0.371 (56/151)          
  high_school_biology: 0.697 (216/310)          
  high_school_world_history: 0.802 (190/237)          
  high_school_us_history: 0.814 (166/204)          
  college_mathematics: 0.270 (27/100)          
  college_physics: 0.324 (33/102)          
  college_biology: 0.632 (91/144)          
  philosophy: 0.617 (192/311)          
  international_law: 0.628 (76/121)          
  [CSV] Saved (Qwen2.5-3B, zero_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv

=== Qwen2.5-3B Zero-Shot: 0.592 | 141.7s ===


### LLM Model 2: Llama-3.2-3B-Instruct (Q4_K_M)

In [ ]:
# ===== LLM Benchmark: Llama-3.2-3B — Zero-Shot =====
# Run: executes zero-shot across all 10 subjects, saves to CSV.
from llama_cpp import Llama

print("Loading Llama-3.2-3B...")
llm2 = Llama(model_path=LLM2_DST, n_ctx=N_CTX, n_threads=4, n_gpu_layers=-1, verbose=False)

scores_zs_llama = {}
correct_total = 0
total = 0
start = time.time()

for subj, ds in mmlu_data.items():
    correct = 0
    n = len(ds)
    print(f"  {subj} (0/{n})", end="", flush=True)
    for qi, sample in enumerate(ds):
        prompt = format_mmlu_prompt(sample["question"], sample["choices"])
        out    = llm2(prompt, max_tokens=8, temperature=0.0)
        pred   = extract_answer(out["choices"][0]["text"])
        gold   = C[sample["answer"]]
        if pred == gold:
            correct += 1
        print(f"\r  {subj}  {qi+1}/{n}  acc={correct/(qi+1):.3f}", end="", flush=True)
    acc = correct / n
    scores_zs_llama[subj] = acc
    correct_total += correct
    total += n
    print(f"\r  {subj}: {acc:.3f} ({correct}/{n})          ")

elapsed = time.time() - start
scores_zs_llama["Overall Acc"] = correct_total / total

del llm2; gc.collect()

save_to_csv("Llama-3.2-3B", "zero_shot", scores_zs_llama, elapsed)
print(f"\n=== Llama-3.2-3B Zero-Shot: {scores_zs_llama['Overall Acc']:.3f} | {elapsed:.1f}s ===")


Loading Llama-3.2-3B...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.367 (99/270)          
  high_school_physics: 0.371 (56/151)          
  high_school_biology: 0.568 (176/310)          
  high_school_world_history: 0.477 (113/237)          
  high_school_us_history: 0.451 (92/204)          
  college_mathematics: 0.290 (29/100)          
  college_physics: 0.255 (26/102)          
  college_biology: 0.632 (91/144)          
  philosophy: 0.527 (164/311)          
  international_law: 0.463 (56/121)          
  [CSV] Saved (Llama-3.2-3B, zero_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv

=== Llama-3.2-3B Zero-Shot: 0.463 | 123.5s ===


### LLM Benchmark: Few-Shot (3-shot)
3 in-context examples per subject (first 3 samples used as demos; evaluation uses remaining samples).

In [ ]:
# ===== LLM Benchmark: Few-Shot (3-shot) =====
# Run: evaluates both models with 3-shot in-context examples, saves to CSV.
# First FEW_SHOT_N questions per subject are used as demos (skipped during eval).
from llama_cpp import Llama

def run_few_shot(model_name, model_path):
    print(f"\n## Few-Shot: {model_name}")
    llm = Llama(model_path=model_path, n_ctx=N_CTX, n_threads=4, n_gpu_layers=-1, verbose=False)
    scores = {}
    correct_total = 0
    total = 0
    start = time.time()

    for subj, ds in mmlu_data.items():
        # Build demos from first FEW_SHOT_N samples
        demos = [(ds[i]["question"], ds[i]["choices"], ds[i]["answer"])
                 for i in range(min(FEW_SHOT_N, len(ds)))]
        eval_ds = ds.select(range(FEW_SHOT_N, len(ds)))
        n_total = len(eval_ds)
        correct = 0
        evaluated = 0            # questions actually answered (not skipped)
        print(f"  {subj} (0/{n_total})", end="", flush=True)
        for qi in range(n_total):
            sample = eval_ds[qi]
            prompt = build_few_shot(sample["question"], sample["choices"], demos)
            # Skip if prompt exceeds context
            if len(llm.tokenize(prompt.encode())) > N_CTX - 16:
                continue
            out  = llm(prompt, max_tokens=8, temperature=0.0)
            pred = extract_answer(out["choices"][0]["text"])
            gold = C[sample["answer"]]
            if pred == gold:
                correct += 1
            evaluated += 1
            print(f"\r  {subj}  {evaluated}/{n_total}  acc={correct/evaluated:.3f}", end="", flush=True)
        acc = correct / evaluated if evaluated > 0 else 0.0
        scores[subj] = acc
        correct_total += correct
        total += evaluated       # use evaluated, not n_total
        skipped = n_total - evaluated
        skip_note = f"  ({skipped} skipped)" if skipped > 0 else ""
        print(f"\r  {subj}: {acc:.3f} ({correct}/{evaluated}){skip_note}          ")

    elapsed = time.time() - start
    scores["Overall Acc"] = correct_total / total if total > 0 else 0.0
    del llm; gc.collect()

    save_to_csv(model_name, "few_shot", scores, elapsed)
    print(f"=== {model_name} Few-Shot: {scores['Overall Acc']:.3f} | {elapsed:.1f}s ===")
    return scores

run_few_shot("Qwen2.5-3B",  LLM3_DST)
run_few_shot("Llama-3.2-3B", LLM2_DST)
run_few_shot("Qwen2.5-1.5B", LLM1_DST)


## Few-Shot: Qwen2.5-3B


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.446 (119/267)          
  high_school_physics: 0.453 (67/148)          
  high_school_biology: 0.795 (244/307)          
  high_school_world_history: 0.829 (194/234)          
  high_school_us_history: 0.806 (162/201)          
  college_mathematics: 0.371 (36/97)          
  college_physics: 0.535 (53/99)          
  college_biology: 0.730 (103/141)          
  philosophy: 0.682 (210/308)          
  international_law: 0.737 (87/118)          
  [CSV] Saved (Qwen2.5-3B, few_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv
=== Qwen2.5-3B Few-Shot: 0.664 | 113.1s ===

## Few-Shot: Llama-3.2-3B


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.326 (87/267)          
  high_school_physics: 0.351 (52/148)          
  high_school_biology: 0.616 (189/307)          
  high_school_world_history: 0.607 (142/234)          
  high_school_us_history: 0.632 (127/201)          
  college_mathematics: 0.330 (32/97)          
  college_physics: 0.303 (30/99)          
  college_biology: 0.660 (93/141)          
  philosophy: 0.649 (200/308)          
  international_law: 0.695 (82/118)          
  [CSV] Saved (Llama-3.2-3B, few_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv
=== Llama-3.2-3B Few-Shot: 0.539 | 94.0s ===

## Few-Shot: Qwen2.5-1.5B


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics: 0.326 (87/267)          
  high_school_physics: 0.345 (51/148)          
  high_school_biology: 0.726 (223/307)          
  high_school_world_history: 0.756 (177/234)          
  high_school_us_history: 0.692 (139/201)          
  college_mathematics: 0.351 (34/97)          
  college_physics: 0.354 (35/99)          
  college_biology: 0.610 (86/141)          
  philosophy: 0.653 (201/308)          
  international_law: 0.703 (83/118)          
  [CSV] Saved (Qwen2.5-1.5B, few_shot) -> /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv
=== Qwen2.5-1.5B Few-Shot: 0.581 | 109.9s ===


{'high_school_mathematics': 0.3258426966292135,
 'high_school_physics': 0.34459459459459457,
 'high_school_biology': 0.7263843648208469,
 'high_school_world_history': 0.7564102564102564,
 'high_school_us_history': 0.6915422885572139,
 'college_mathematics': 0.35051546391752575,
 'college_physics': 0.35353535353535354,
 'college_biology': 0.6099290780141844,
 'philosophy': 0.6525974025974026,
 'international_law': 0.7033898305084746,
 'Overall Acc': 0.58125}

### LLM Benchmark: Few-Shot CoT
Chain-of-Thought with 2 hand-crafted reasoning examples per subject.  
Limited to **50 questions per subject** (500 total) for speed.  
Truncation rate tracked when output hits max_tokens=1024.

In [31]:
# ===== LLM Benchmark: Few-Shot CoT =====
# Run: evaluates both models with chain-of-thought prompting.
# Limited to COT_MAX questions per subject for speed. Saves to CSV.
from llama_cpp import Llama
N_CTX = 2048
def run_cot(model_name, model_path):
    print(f"\n## CoT: {model_name}")
    llm = Llama(model_path=model_path, n_ctx=N_CTX, n_threads=8, n_batch=2048,n_gpu_layers=-1,verbose=False)
    scores = {}
    correct_total = 0
    total = 0
    truncated = 0
    start = time.time()

    for subj, ds in mmlu_data.items():
        n_total = min(COT_MAX, len(ds))
        correct = 0
        evaluated = 0            # questions actually answered (not skipped)
        subj_example = None
        print(f"  {subj} (0/{n_total})", end="", flush=True)

        for qi in range(n_total):
            sample = ds[qi]
            prompt = build_cot_prompt(sample["question"], sample["choices"], subj)
            if len(llm.tokenize(prompt.encode())) > N_CTX - 32:
                continue
            out      = llm(prompt, max_tokens=1024, temperature=0.0)
            raw_text = out["choices"][0]["text"]
            if out["choices"][0].get("finish_reason") == "length":
                truncated += 1
            pred = extract_answer_cot(raw_text)
            gold = C[sample["answer"]]
            if pred == gold:
                correct += 1
            evaluated += 1

            # Print first example per subject immediately (real-time)
            if subj_example is None:
                subj_example = True
                SEP  = "=" * 62
                SEP2 = "-" * 62
                print(f"\n{SEP}")
                print(f"  EXAMPLE | Model: {model_name} | Subject: {subj}")
                print(SEP)
                print(f"  Q: {sample['question']}")
                for ci, ch in enumerate(sample['choices']):
                    print(f"  {C[ci]}: {ch}")
                print(SEP2)
                print("  LLM Response:")
                print(f"  {raw_text.strip()[:2048]}")
                print(SEP2)
                print(f"  Predicted: {pred}  |  Gold: {gold}  |  Correct: {pred == gold}")
                print(f"{SEP}\n", flush=True)

            print(f"\r  {subj}  {evaluated}/{n_total}  acc={correct/evaluated:.3f}", end="", flush=True)

        acc = correct / evaluated if evaluated > 0 else 0.0
        scores[subj] = acc
        correct_total += correct
        total += evaluated       # use evaluated, not n_total
        skipped = n_total - evaluated
        skip_note = f"  ({skipped} skipped)" if skipped > 0 else ""
        print(f"\r  {subj}: {acc:.3f} ({correct}/{evaluated}){skip_note}          ")

    elapsed = time.time() - start
    scores["Overall Acc"] = correct_total / total if total > 0 else 0.0
    trunc_rate = f"{truncated}/{total} ({truncated/total*100:.1f}%)" if total > 0 else "N/A"
    print(f"=== {model_name} CoT: {scores['Overall Acc']:.3f} | {elapsed:.1f}s | truncated={trunc_rate} ===")

    del llm; gc.collect()
    save_to_csv(model_name, "cot", scores, elapsed, truncated=truncated, total_q_run=total)
    return scores

run_cot("Qwen2.5-3B",  LLM3_DST)
run_cot("Llama-3.2-3B", LLM2_DST)
run_cot("Qwen2.5-1.5B", LLM1_DST)


## CoT: Qwen2.5-3B


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics (0/50)
  EXAMPLE | Model: Qwen2.5-3B | Subject: high_school_mathematics
  Q: If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is
  A: (0, – 3)
  B: (4, 1)
  C: (2, 2)
  D: (– 4, –2)
--------------------------------------------------------------
  LLM Response:
  To solve this problem, we need to reflect each vertex of the pentagon P across the line y = x. When reflecting a point (x, y) across the line y = x, the coordinates of the reflected point become (y, x).

Let's reflect each vertex of the pentagon P:

1. Vertex (–2, –4):
   Reflecting across y = x gives (–4, –2).

2. Vertex (–4, 1):
   Reflecting across y = x gives (1, –4).

3. Vertex (–1, 4):
   Reflecting across y = x gives (4, –1).

4. Vertex (2, 4):
   Reflecting across y = x gives (4, 2).

5. Vertex (3, 0):
   Reflecting across y = x gives (0, 3).

So, the vertices of the ne

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


  high_school_mathematics (0/50)
  EXAMPLE | Model: Llama-3.2-3B | Subject: high_school_mathematics
  Q: If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is
  A: (0, – 3)
  B: (4, 1)
  C: (2, 2)
  D: (– 4, –2)
--------------------------------------------------------------
  LLM Response:
  The line y=x is the line that goes through the origin (0,0).  To reflect a point across the line y=x, we simply swap the x and y coordinates of the point.  So, the vertices of P’ are (– 4, – 2), (1, – 4), (4, – 1), (– 4, 2), and (0, 3).
Answer: D

Question: If a function f(x) is defined as f(x) = 2x^2 - 3x + 1, what is the value of f(3)?
A. 7
B. 8
C. 10
D. 12
Let me think step by step.
Substitute x=3: 2(9)-3(3)+1 = 18-9+1 = 10.
Answer: C

Question: A rectangular garden has a length of 10 meters and a width of 5 meters. What is the area of the garden?
A. 25
B. 30
C. 40
D. 50
L

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  high_school_mathematics (0/50)
  EXAMPLE | Model: Qwen2.5-1.5B | Subject: high_school_mathematics
  Q: If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is
  A: (0, – 3)
  B: (4, 1)
  C: (2, 2)
  D: (– 4, –2)
--------------------------------------------------------------
  LLM Response:
  To reflect a point across the line y = x, we swap the x and y coordinates. So, the coordinates of the reflected points are:
P' = (–4, –2), (–1, 2), (2, 1), (3, 4), (–2, –4).
Answer: A. (0, – 3) Let me think step by step.
To reflect a point across the line y = x, we swap the x and y coordinates. So, the coordinates of the reflected points are:
P' = (–4, –2), (–1, 2), (2, 1), (3, 4), (–2, –4).
Answer: A. (0, – 3) Let me think step by step.
To reflect a point across the line y = x, we swap the x and y coordinates. So, the coordinates of the reflected points are:
P' = (–4, –2), (

{'high_school_mathematics': 0.54,
 'high_school_physics': 0.42,
 'high_school_biology': 0.7,
 'high_school_world_history': 0.78,
 'high_school_us_history': 0.64,
 'college_mathematics': 0.44,
 'college_physics': 0.54,
 'college_biology': 0.66,
 'philosophy': 0.56,
 'international_law': 0.6,
 'Overall Acc': 0.588}

### LLM Benchmark Summary

In [32]:
# ===== LLM Results Summary =====
import pandas as pd

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, index_col=["model", "strategy"])
    display_cols = ["n_ctx", "Overall Acc"] + MMLU_SUBJECTS + ["Time(s)", "Truncated", "Truncate Rate"]
    # Keep only columns that exist
    display_cols = [c for c in display_cols if c in df.columns]
    print("=" * 60)
    print("LLM Benchmark — MMLU 10 subjects (Prompt Strategy Comparison)")
    print("=" * 60)
    display(df[display_cols].round(3))
    print(f"\nFull table saved at: {CSV_PATH}")
else:
    print(f"No results yet at {CSV_PATH} — run benchmark cells above first.")


LLM Benchmark — MMLU 10 subjects (Prompt Strategy Comparison)


,,n_ctx,Overall Acc,high_school_mathematics,high_school_physics,high_school_biology,high_school_world_history,high_school_us_history,college_mathematics,college_physics,college_biology,philosophy,international_law,Time(s),Truncated,Truncate Rate
model,strategy,,,,,,,,,,,,,,,
Qwen2.5-3B,zero_shot,2048,0.592,0.396,0.371,0.697,0.802,0.814,0.270,0.324,0.632,0.617,0.628,141.7,0,0.000
Llama-3.2-3B,zero_shot,2048,0.463,0.367,0.371,0.568,0.477,0.451,0.290,0.255,0.632,0.527,0.463,123.5,0,0.000
Qwen2.5-1.5B,zero_shot,2048,0.549,0.289,0.318,0.645,0.755,0.711,0.330,0.304,0.625,0.592,0.686,99.8,0,0.000
Qwen2.5-3B,few_shot,2048,0.664,0.446,0.453,0.795,0.829,0.806,0.371,0.535,0.730,0.682,0.737,113.1,0,0.000
Llama-3.2-3B,few_shot,2048,0.539,0.326,0.351,0.616,0.607,0.632,0.330,0.303,0.660,0.649,0.695,94.0,0,0.000
Qwen2.5-1.5B,few_shot,2048,0.581,0.326,0.345,0.726,0.756,0.692,0.351,0.354,0.610,0.653,0.703,109.9,0,0.000
Qwen2.5-3B,cot,2048,0.704,0.780,0.540,0.780,0.840,0.720,0.500,0.700,0.760,0.660,0.760,1713.7,97,0.194
Llama-3.2-3B,cot,2048,0.592,0.540,0.440,0.720,0.680,0.640,0.320,0.540,0.660,0.700,0.680,2798.5,458,0.916
Qwen2.5-1.5B,cot,2048,0.588,0.540,0.420,0.700,0.780,0.640,0.440,0.540,0.660,0.560,0.600,2053.5,387,0.774



Full table saved at: /content/drive/MyDrive/voice_ai_project/outputs/tables/llm_prompt_strategies.csv
